# 08 — Backtest (walk 1, 2009 test window, all strategies)

Apples-to-apples comparison of 8 portfolio strategies on the walk-1 2009 holdout
under standardized 5 bps round-trip execution cost. Resumable / gracefully
handles missing PPO variants: only backtests whichever PPO cost variants have
a `best_model.zip` saved at run time.

**Strategies**: EW-Top30, Score-Prop, MinVar, 60/40 (proxy), PPO-{5,2,10,20}bps.
**Metrics**: total/annualized return, vol, Sharpe, Sortino, max DD, Calmar, hit rate, turnover.

**Spec:** `docs/superpowers/specs/2026-05-17-backtest-design.md`.

## A. Setup + load whatever's available

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor

from src.utils.io import processed_dir, repo_root
from src.utils.ranker import friday_only, project_text_to_pca, load_walk_pca
from src.utils.rl_env import PortfolioEnv, project_to_simplex, build_scoreboard_from_scored_panel
from src.utils.backtest import (
    compute_strategy_metrics,
    equal_weight_weights,
    min_variance_weights,
    score_proportional_weights,
)

WALK_ID = 1
TEST_START, TEST_END = '2009-01-01', '2009-12-31'

TOP_K = 30
EPISODE_LENGTH = 52      # 52 Fridays = full 2009
MAX_WEIGHT = 1.0         # no per-stock cap — apples-to-apples with PPO action expansion
COST_BPS = 5.0           # standardized execution cost for all strategies
MINVAR_LOOKBACK = 26     # weeks of trailing returns for MinVar cov estimate
RANDOM_STATE = 42

PPO_COSTS_TO_BACKTEST = [5, 2, 10, 20]  # graceful skip if any missing

PANEL_DIR  = processed_dir() / 'training_panel'
EMBED_DIR  = processed_dir() / 'finbert_stockday_embed'
RANKER_DIR = repo_root() / 'artifacts' / 'ranker' / f'walk-{WALK_ID:03d}'
RL_DIR     = repo_root() / 'artifacts' / 'rl' / f'walk-{WALK_ID:03d}'
OUT_DIR    = repo_root() / 'artifacts' / 'backtest' / f'walk-{WALK_ID:03d}'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# We need a scoreboard covering history (for MinVar lookback) AND the test window.
HISTORY_START = pd.Timestamp(TEST_START) - pd.Timedelta(weeks=MINVAR_LOOKBACK + 4)

# Load the existing scoreboard (built by notebook 07 for train+val) and extend
# it with whatever's needed to cover [HISTORY_START, TEST_END].
SCOREBOARD_PATH = RL_DIR / 'scoreboard.parquet'
scoreboard_all = pd.read_parquet(SCOREBOARD_PATH)
scoreboard_all['date'] = pd.to_datetime(scoreboard_all['date'])
existing_max = scoreboard_all['date'].max()
print(f'existing scoreboard: {scoreboard_all["date"].min().date()} .. {existing_max.date()}')

if existing_max < pd.Timestamp(TEST_END):
    print(f'extending scoreboard to cover {TEST_START}..{TEST_END}...')
    ranker_bundle = joblib.load(RANKER_DIR / 'model.joblib')
    ranker_model = ranker_bundle['model']
    ranker_features = ranker_bundle['feature_names']
    pca, n_pca = load_walk_pca(WALK_ID)

    extend_start = (existing_max + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    extend_end = TEST_END
    print(f'  loading panel + embed for {extend_start}..{extend_end}')

    def _load_years(dir_, start, end, cols=None):
        s, e = pd.Timestamp(start), pd.Timestamp(end)
        frames = []
        for y in range(s.year, e.year + 1):
            for p in sorted((dir_ / f'year={y}').glob('*.parquet')):
                df = pd.read_parquet(p, columns=cols)
                df['date'] = pd.to_datetime(df['date'])
                df = df[(df['date'] >= s) & (df['date'] <= e)]
                frames.append(df)
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    panel_ext = _load_years(PANEL_DIR, extend_start, extend_end)
    embed_ext = _load_years(EMBED_DIR, extend_start, extend_end,
                            cols=['permno', 'date', 'vec'])
    embed_pca_ext = project_text_to_pca(embed_ext, pca)
    fri_ext = friday_only(panel_ext).merge(embed_pca_ext, on=['permno', 'date'], how='inner')
    fri_ext = fri_ext.dropna(subset=['fwd_ret_5d']).copy()

    X_ext = pd.DataFrame({c: fri_ext[c] if c in fri_ext.columns else np.nan
                          for c in ranker_features})
    fri_ext['score'] = ranker_model.predict(X_ext)
    new_scores = build_scoreboard_from_scored_panel(fri_ext, top_k=TOP_K)
    print(f'  built {len(new_scores)} new rows ({new_scores["date"].nunique()} Fridays)')

    scoreboard_all = pd.concat([scoreboard_all, new_scores], ignore_index=True)
    scoreboard_all = scoreboard_all.sort_values(['date', 'permno']).reset_index(drop=True)
    scoreboard_all.to_parquet(SCOREBOARD_PATH, compression='zstd', index=False)
    print(f'  extended scoreboard written -> {SCOREBOARD_PATH.relative_to(repo_root())}')
else:
    print('scoreboard already covers test window — no extension needed')

# Slice test window for the backtest loop, and history+test for MinVar.
scoreboard_test = scoreboard_all[
    (scoreboard_all['date'] >= TEST_START) & (scoreboard_all['date'] <= TEST_END)
].copy().reset_index(drop=True)
scoreboard_with_history = scoreboard_all[
    (scoreboard_all['date'] >= HISTORY_START) & (scoreboard_all['date'] <= TEST_END)
].copy().reset_index(drop=True)
test_dates = sorted(scoreboard_test['date'].unique())
print(f'\ntest window: {test_dates[0].date()} .. {test_dates[-1].date()}  '
      f'({len(test_dates)} Fridays)')
print(f'history+test (for MinVar): {scoreboard_with_history["date"].min().date()} .. '
      f'{scoreboard_with_history["date"].max().date()}')

# Detect which PPO cost variants are actually available.
available_ppo = []
for cb in PPO_COSTS_TO_BACKTEST:
    best_path = RL_DIR / f'cost-{cb:03d}bps' / 'best_model.zip'
    norm_path = RL_DIR / f'cost-{cb:03d}bps' / 'vec_normalize.pkl'
    if best_path.exists() and norm_path.exists():
        available_ppo.append(cb)
        print(f'  PPO-{cb}bps: ✓ found')
    else:
        print(f'  PPO-{cb}bps: ✗ missing (skipping)')
print(f'\nwill backtest: 4 baselines + {len(available_ppo)} PPO variants = '
      f'{4 + len(available_ppo)} strategies')

## B. Strategy runner (deterministic 2009 playthrough)

In [ ]:
# Group scoreboard by date for fast lookup.
by_date_test = {d: g.reset_index(drop=True) for d, g in scoreboard_test.groupby('date')}
by_date_history = {d: g.reset_index(drop=True)
                   for d, g in scoreboard_with_history.groupby('date')}
dates_with_history = sorted(by_date_history.keys())
test_dates_sorted = sorted(by_date_test.keys())


def _trailing_returns_matrix(date, lookback: int) -> np.ndarray:
    """For MinVar: build (lookback, TOP_K) trailing fwd_ret_5d matrix anchored
    at `date`'s top-K permnos. Stocks not in prior weeks → NaN → zeroed in helper."""
    if date not in by_date_test:
        return np.zeros((lookback, TOP_K), dtype=np.float32)
    cur_permnos = by_date_test[date]['permno'].tolist()[:TOP_K]
    prior = [d for d in dates_with_history if d < date][-lookback:]
    if len(prior) < lookback // 2:
        return None
    rows = []
    for d in prior:
        g = by_date_history[d].set_index('permno')['fwd_ret_5d']
        row = g.reindex(cur_permnos).to_numpy(dtype=np.float32)
        rows.append(row)
    return np.vstack(rows)


def run_one_strategy(name: str, weights_fn) -> dict:
    """Deterministic 2009 playthrough. `weights_fn(date, cur_df, prev_weights, last_return)`
    returns the next weight vector (length TOP_K, sum 1)."""
    prev_weights = equal_weight_weights(TOP_K)
    last_return = 0.0  # tracked and passed to weights_fn so PPO obs matches training
    weekly_returns = []
    weekly_turnover = []
    weekly_dates = []

    for date in test_dates_sorted:
        cur = by_date_test[date]
        new_weights = weights_fn(date, cur, prev_weights, last_return)
        rets = np.nan_to_num(cur['fwd_ret_5d'].to_numpy(dtype=np.float32)[:TOP_K], nan=0.0)
        port_ret = float(np.dot(new_weights, rets))
        turnover = float(np.abs(new_weights - prev_weights).sum())
        weekly_returns.append(port_ret)
        weekly_turnover.append(turnover)
        weekly_dates.append(date)
        prev_weights = new_weights
        last_return = port_ret  # now the next iter sees the real last_return

    weekly_returns = np.array(weekly_returns)
    weekly_turnover = np.array(weekly_turnover)
    metrics = compute_strategy_metrics(weekly_returns, weekly_turnover, cost_bps=COST_BPS)
    metrics['strategy'] = name
    cost_per_wk = (COST_BPS / 10_000.0) * weekly_turnover
    net_returns = weekly_returns - cost_per_wk
    return {
        'name': name,
        'metrics': metrics,
        'weekly_dates': weekly_dates,
        'weekly_returns_gross': weekly_returns,
        'weekly_returns_net': net_returns,
        'weekly_turnover': weekly_turnover,
    }


# --- weight functions for each strategy (all take last_return for consistent signature) ---

def w_equal(date, cur, prev_w, last_ret):
    return equal_weight_weights(TOP_K)


def w_score_prop(date, cur, prev_w, last_ret):
    scores = cur['score'].to_numpy(dtype=np.float32)[:TOP_K]
    return score_proportional_weights(scores, max_weight=MAX_WEIGHT)


def w_minvar(date, cur, prev_w, last_ret):
    history = _trailing_returns_matrix(date, MINVAR_LOOKBACK)
    if history is None or np.isnan(history).all():
        return equal_weight_weights(TOP_K)
    try:
        return min_variance_weights(history, max_weight=MAX_WEIGHT)
    except Exception as e:
        print(f'  MinVar fallback at {date.date()}: {e}')
        return equal_weight_weights(TOP_K)


def run_60_40() -> dict:
    """Special case: 60/40 needs bond return added back."""
    prev_weights = equal_weight_weights(TOP_K)
    weekly_returns = []
    weekly_turnover = []
    weekly_dates = []

    for date in test_dates_sorted:
        cur = by_date_test[date]
        new_weights = equal_weight_weights(TOP_K)
        rets = np.nan_to_num(cur['fwd_ret_5d'].to_numpy(dtype=np.float32)[:TOP_K], nan=0.0)
        eq_ret = float(np.dot(new_weights, rets))

        bond_yield = float(cur['macro_dgs10'].iloc[0]) if 'macro_dgs10' in cur.columns else 4.0
        bond_ret = bond_yield / 5200.0

        port_ret = 0.6 * eq_ret + 0.4 * bond_ret
        turnover = 0.6 * float(np.abs(new_weights - prev_weights).sum())

        weekly_returns.append(port_ret)
        weekly_turnover.append(turnover)
        weekly_dates.append(date)
        prev_weights = new_weights

    weekly_returns = np.array(weekly_returns)
    weekly_turnover = np.array(weekly_turnover)
    metrics = compute_strategy_metrics(weekly_returns, weekly_turnover, cost_bps=COST_BPS)
    metrics['strategy'] = '60/40 (proxy)'
    cost_per_wk = (COST_BPS / 10_000.0) * weekly_turnover
    return {
        'name': '60/40 (proxy)',
        'metrics': metrics,
        'weekly_dates': weekly_dates,
        'weekly_returns_gross': weekly_returns,
        'weekly_returns_net': weekly_returns - cost_per_wk,
        'weekly_turnover': weekly_turnover,
    }


def make_ppo_weights_fn(cost_bps: int):
    """Load best_model + vec_normalize for a cost variant, return a weights_fn."""
    cv_dir = RL_DIR / f'cost-{cost_bps:03d}bps'
    model = PPO.load(cv_dir / 'best_model.zip')

    def _ppo_env_fn():
        env = PortfolioEnv(scoreboard=scoreboard_test, top_k=TOP_K,
                           episode_length=EPISODE_LENGTH, cost_bps=float(cost_bps),
                           max_weight=MAX_WEIGHT)
        env.reset(seed=RANDOM_STATE)
        return Monitor(env)

    vec = DummyVecEnv([_ppo_env_fn])
    vec = VecNormalize.load(str(cv_dir / 'vec_normalize.pkl'), vec)
    vec.training = False
    vec.norm_reward = False

    def w_ppo(date, cur, prev_w, last_ret):
        # Construct a temporary env, position it at this date, set the previous
        # weights AND the actual last return so the observation matches what
        # PPO saw during training (the `last_return` slot in obs[-1]).
        tmp = PortfolioEnv(scoreboard=scoreboard_test, top_k=TOP_K,
                           episode_length=EPISODE_LENGTH, cost_bps=float(cost_bps),
                           max_weight=MAX_WEIGHT)
        tmp.reset(seed=0)
        idx = tmp._dates.tolist().index(date)
        tmp._idx = idx
        tmp._weights = np.asarray(prev_w, dtype=np.float32)
        tmp._last_return = float(last_ret)
        obs = tmp._build_obs()
        obs_norm = vec.normalize_obs(obs)
        action, _ = model.predict(obs_norm, deterministic=True)
        return project_to_simplex(np.asarray(action, dtype=np.float64), max_weight=MAX_WEIGHT)

    return w_ppo

## C. Run all available strategies

In [ ]:
results = []

print('running baselines...')
results.append(run_one_strategy('EW-Top30',  w_equal))
print('  EW-Top30 done')
results.append(run_one_strategy('Score-Prop', w_score_prop))
print('  Score-Prop done')
results.append(run_one_strategy('MinVar',    w_minvar))
print('  MinVar done')
results.append(run_60_40())
print('  60/40 (proxy) done')

print(f'\nrunning {len(available_ppo)} PPO variant(s)...')
for cb in available_ppo:
    name = f'PPO-{cb}bps'
    print(f'  loading {name}...')
    ppo_fn = make_ppo_weights_fn(cb)
    results.append(run_one_strategy(name, ppo_fn))
    print(f'  {name} done')

print(f'\nfinished {len(results)} strategies')

## D. Metrics table

In [ ]:
summary_rows = []
for r in results:
    row = {'strategy': r['name'], **r['metrics']}
    row.pop('strategy', None)  # avoid dupe key
    row = {'strategy': r['name'], **{k: v for k, v in r['metrics'].items() if k != 'strategy'}}
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
# Round and order for display.
display_cols = [
    'strategy', 'total_return_net', 'annualized_return', 'annualized_vol',
    'sharpe', 'sortino', 'max_drawdown', 'calmar', 'hit_rate', 'avg_turnover',
]
display_df = summary_df[display_cols].copy()
for c in display_cols[1:]:
    display_df[c] = display_df[c].round(4)
print('=== backtest summary (walk 1, 2009 test, 5 bps execution cost) ===')
print(display_df.to_string(index=False))

## E. Equity curves + drawdown + Sharpe bars

In [ ]:
# --- Equity curves ---
fig, ax = plt.subplots(figsize=(10, 5.5))
for r in results:
    eq = np.cumprod(1.0 + r['weekly_returns_net'])
    ax.plot(r['weekly_dates'], eq, label=r['name'], lw=1.6)
ax.axhline(1.0, color='black', lw=0.5, ls='--', alpha=0.5)
ax.set_xlabel('date')
ax.set_ylabel('cumulative net return (1.0 = breakeven)')
ax.set_title(f'Equity curves — walk 1 test (2009), {COST_BPS} bps execution cost')
ax.legend(loc='best', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'equity_curves.png', dpi=120)
plt.show()

# --- Drawdown curves ---
fig, ax = plt.subplots(figsize=(10, 4.5))
for r in results:
    eq = np.cumprod(1.0 + r['weekly_returns_net'])
    dd = eq / np.maximum.accumulate(eq) - 1.0
    ax.plot(r['weekly_dates'], dd, label=r['name'], lw=1.4)
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('date')
ax.set_ylabel('drawdown')
ax.set_title('Drawdown curves — walk 1 test (2009)')
ax.legend(loc='best', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'drawdown.png', dpi=120)
plt.show()

# --- Sharpe bar chart ---
fig, ax = plt.subplots(figsize=(8, 4.5))
sharpes = summary_df.set_index('strategy')['sharpe']
colors = ['steelblue' if 'PPO' not in s else 'darkorange' for s in sharpes.index]
ax.bar(range(len(sharpes)), sharpes.values, color=colors)
ax.set_xticks(range(len(sharpes)))
ax.set_xticklabels(sharpes.index, rotation=30, ha='right')
ax.axhline(0, color='black', lw=0.5)
ax.set_ylabel('annualized Sharpe (rf=0)')
ax.set_title('Sharpe across strategies — walk 1 test (2009)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'sharpe_bars.png', dpi=120)
plt.show()

## F. Persist all artifacts

In [ ]:
# Per-week returns long-format parquet for full reproducibility.
weekly_rows = []
for r in results:
    for d, gr, nr, tu in zip(r['weekly_dates'], r['weekly_returns_gross'],
                              r['weekly_returns_net'], r['weekly_turnover']):
        weekly_rows.append({
            'strategy': r['name'], 'date': d,
            'return_gross': float(gr), 'return_net': float(nr),
            'turnover': float(tu),
        })
weekly_df = pd.DataFrame(weekly_rows)
weekly_df.to_parquet(OUT_DIR / 'weekly_returns.parquet', compression='zstd', index=False)

# Summary CSV.
summary_df.to_csv(OUT_DIR / 'summary_table.csv', index=False)

# Per-strategy metrics JSON.
per_strategy = {r['name']: {k: float(v) if isinstance(v, (int, float, np.floating)) else v
                            for k, v in r['metrics'].items()}
                for r in results}
(OUT_DIR / 'per_strategy_metrics.json').write_text(json.dumps(per_strategy, indent=2))

print(f'wrote artifacts to {OUT_DIR.relative_to(repo_root())}/:')
for p in sorted(OUT_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size / 1024:.1f} KB)')

## G. Multi-walk concatenated backtest (full OOS 2009-2024)

Loops walks 1-16. For each walk: backtest baselines (EW, Score-Prop, MinVar,
60/40) + PPO on that walk's test year (2009+N-1). Also includes no-text
variants (EW-NoText, Score-NoText, PPO-NoText) if `artifacts/no-text/` is
populated by notebook 09. Per-walk weekly returns concatenated → one 16-year
return series per strategy → metrics computed on the full series.

This is the paper's headline figure: equity curves across 16 years OOS,
text-enhanced vs no-text vs naive baselines.

In [ ]:
WALK_RANGE_BT = range(1, 17)
NO_TEXT_ROOT = repo_root() / 'artifacts' / 'no-text'


def _walk_artifacts(walk_id: int) -> dict:
    rl_dir = repo_root() / 'artifacts' / 'rl' / f'walk-{walk_id:03d}'
    nt_dir = NO_TEXT_ROOT / 'rl' / f'walk-{walk_id:03d}'
    nt_walk_dir = NO_TEXT_ROOT / f'walk-{walk_id:03d}'
    return {
        'full_scoreboard':   rl_dir / 'scoreboard.parquet',
        'full_ppo':          rl_dir / f'cost-{int(COST_BPS):03d}bps' / 'best_model.zip',
        'full_vec_norm':     rl_dir / f'cost-{int(COST_BPS):03d}bps' / 'vec_normalize.pkl',
        'no_text_scoreboard':nt_walk_dir / 'scoreboard.parquet',
        'no_text_ppo':       nt_dir / f'cost-{int(COST_BPS):03d}bps' / 'best_model.zip',
        'no_text_vec_norm':  nt_dir / f'cost-{int(COST_BPS):03d}bps' / 'vec_normalize.pkl',
    }


def _run_strategies_on_scoreboard(sb_test: pd.DataFrame, sb_history: pd.DataFrame,
                                  variant: str, ppo_paths: dict | None = None) -> dict:
    """Run the strategy menu on a single walk's scoreboard slice.
    `variant` ∈ {'full', 'no_text'}. Returns dict of {name: dict_with_weekly_returns}."""
    by_test = {d: g.reset_index(drop=True) for d, g in sb_test.groupby('date')}
    by_hist = {d: g.reset_index(drop=True) for d, g in sb_history.groupby('date')}
    dates_hist = sorted(by_hist.keys())
    dates_test = sorted(by_test.keys())

    def _trailing_returns(date, lookback):
        if date not in by_test:
            return None
        cur_permnos = by_test[date]['permno'].tolist()[:TOP_K]
        prior = [d for d in dates_hist if d < date][-lookback:]
        if len(prior) < lookback // 2:
            return None
        rows = []
        for d in prior:
            g = by_hist[d].set_index('permno')['fwd_ret_5d']
            row = g.reindex(cur_permnos).to_numpy(dtype=np.float32)
            rows.append(row)
        return np.vstack(rows)

    def _runner(name, weights_fn, use_60_40: bool = False):
        prev_w = equal_weight_weights(TOP_K)
        last_ret = 0.0
        rets, turn, dates_out = [], [], []
        for date in dates_test:
            cur = by_test[date]
            new_w = weights_fn(date, cur, prev_w, last_ret)
            rs = np.nan_to_num(cur['fwd_ret_5d'].to_numpy(dtype=np.float32)[:TOP_K], nan=0.0)
            if use_60_40:
                eq_ret = float(np.dot(new_w, rs))
                bond_yield = float(cur['macro_dgs10'].iloc[0]) if 'macro_dgs10' in cur.columns else 4.0
                port_ret = 0.6 * eq_ret + 0.4 * (bond_yield / 5200.0)
                turnover = 0.6 * float(np.abs(new_w - prev_w).sum())
            else:
                port_ret = float(np.dot(new_w, rs))
                turnover = float(np.abs(new_w - prev_w).sum())
            rets.append(port_ret)
            turn.append(turnover)
            dates_out.append(date)
            prev_w = new_w
            last_ret = port_ret
        return {'name': name, 'dates': dates_out,
                'returns_gross': np.array(rets), 'turnover': np.array(turn)}

    suffix = '' if variant == 'full' else '-NoText'
    results = {
        f'EW-Top30{suffix}':   _runner(f'EW-Top30{suffix}',
                                       lambda d, c, pw, lr: equal_weight_weights(TOP_K)),
        f'Score-Prop{suffix}': _runner(f'Score-Prop{suffix}',
                                       lambda d, c, pw, lr: score_proportional_weights(
                                           c['score'].to_numpy(dtype=np.float32)[:TOP_K],
                                           max_weight=MAX_WEIGHT)),
    }
    # MinVar and 60/40 only on the 'full' variant (universe-agnostic in spirit).
    if variant == 'full':
        def _minvar_fn(date, cur, prev_w, last_ret):
            hist = _trailing_returns(date, MINVAR_LOOKBACK)
            if hist is None or np.isnan(hist).all():
                return equal_weight_weights(TOP_K)
            try:
                return min_variance_weights(hist, max_weight=MAX_WEIGHT)
            except Exception:
                return equal_weight_weights(TOP_K)
        results['MinVar'] = _runner('MinVar', _minvar_fn)
        results['60/40 (proxy)'] = _runner('60/40 (proxy)',
                                           lambda d, c, pw, lr: equal_weight_weights(TOP_K),
                                           use_60_40=True)

    # PPO if model artifact present.
    if ppo_paths is not None and ppo_paths['model'].exists():
        model = PPO.load(ppo_paths['model'])
        def _env_fn():
            env = PortfolioEnv(scoreboard=sb_test, top_k=TOP_K,
                               episode_length=EPISODE_LENGTH, cost_bps=float(COST_BPS),
                               max_weight=MAX_WEIGHT)
            env.reset(seed=RANDOM_STATE)
            return Monitor(env)
        vec = DummyVecEnv([_env_fn])
        vec = VecNormalize.load(str(ppo_paths['vec_norm']), vec)
        vec.training = False
        vec.norm_reward = False

        def _ppo_fn(date, cur, prev_w, last_ret):
            tmp = PortfolioEnv(scoreboard=sb_test, top_k=TOP_K,
                               episode_length=EPISODE_LENGTH, cost_bps=float(COST_BPS),
                               max_weight=MAX_WEIGHT)
            tmp.reset(seed=0)
            tmp._idx = tmp._dates.tolist().index(date)
            tmp._weights = np.asarray(prev_w, dtype=np.float32)
            tmp._last_return = float(last_ret)
            obs = tmp._build_obs()
            obs_norm = vec.normalize_obs(obs)
            action, _ = model.predict(obs_norm, deterministic=True)
            return project_to_simplex(np.asarray(action, dtype=np.float64), max_weight=MAX_WEIGHT)
        results[f'PPO{suffix}'] = _runner(f'PPO{suffix}', _ppo_fn)
    return results


# Run for each walk; concat into long per-strategy series.
multi_walk_results: dict[str, dict] = {}  # strategy_name -> {'dates': [...], 'returns_gross': [...], 'turnover': [...]}

for w in WALK_RANGE_BT:
    test_year = 2009 + w - 1
    test_start_w = f'{test_year}-01-01'
    test_end_w = f'{test_year}-12-31'
    history_start_w = pd.Timestamp(test_start_w) - pd.Timedelta(weeks=MINVAR_LOOKBACK + 4)

    arts = _walk_artifacts(w)
    if not arts['full_scoreboard'].exists():
        print(f'walk {w}: no full scoreboard ({arts["full_scoreboard"]}) — skipping')
        continue

    # FULL-TEXT variant
    sb_full = pd.read_parquet(arts['full_scoreboard'])
    sb_full['date'] = pd.to_datetime(sb_full['date'])
    sb_test_f = sb_full[(sb_full['date'] >= test_start_w) & (sb_full['date'] <= test_end_w)]
    sb_hist_f = sb_full[(sb_full['date'] >= history_start_w) & (sb_full['date'] <= test_end_w)]
    if len(sb_test_f) == 0:
        print(f'walk {w}: no rows in test window ({test_year}) — skipping')
        continue

    full_ppo_paths = ({'model': arts['full_ppo'], 'vec_norm': arts['full_vec_norm']}
                      if arts['full_ppo'].exists() else None)
    full_res = _run_strategies_on_scoreboard(sb_test_f, sb_hist_f, 'full', full_ppo_paths)

    # NO-TEXT variant (optional)
    nt_res = {}
    if arts['no_text_scoreboard'].exists():
        sb_nt = pd.read_parquet(arts['no_text_scoreboard'])
        sb_nt['date'] = pd.to_datetime(sb_nt['date'])
        sb_test_nt = sb_nt[(sb_nt['date'] >= test_start_w) & (sb_nt['date'] <= test_end_w)]
        sb_hist_nt = sb_nt[(sb_nt['date'] >= history_start_w) & (sb_nt['date'] <= test_end_w)]
        nt_ppo_paths = ({'model': arts['no_text_ppo'], 'vec_norm': arts['no_text_vec_norm']}
                        if arts['no_text_ppo'].exists() else None)
        nt_res = _run_strategies_on_scoreboard(sb_test_nt, sb_hist_nt, 'no_text', nt_ppo_paths)

    all_res = {**full_res, **nt_res}
    n_ppo = sum(1 for k in all_res if k.startswith('PPO'))
    print(f'walk {w} ({test_year}): ran {len(all_res)} strategies '
          f'({n_ppo} PPO variant(s))')

    for name, r in all_res.items():
        if name not in multi_walk_results:
            multi_walk_results[name] = {'dates': [], 'returns_gross': [], 'turnover': []}
        multi_walk_results[name]['dates'].extend(r['dates'])
        multi_walk_results[name]['returns_gross'].extend(r['returns_gross'].tolist())
        multi_walk_results[name]['turnover'].extend(r['turnover'].tolist())

print(f'\nconcat done: {len(multi_walk_results)} strategies, '
      f'{len(next(iter(multi_walk_results.values()))["dates"])} total weeks per strategy')

## H. Full-period metrics + equity curves (the paper's headline figure)

Compute Sharpe / Sortino / MDD / Calmar / hit-rate / turnover on the FULL
concatenated 16-year series per strategy. Sort by Sharpe. Save table + plot.

In [ ]:
mw_summary_rows = []
mw_weekly_rows = []

for name, r in multi_walk_results.items():
    returns_gross = np.array(r['returns_gross'])
    turnover = np.array(r['turnover'])
    m = compute_strategy_metrics(returns_gross, turnover, cost_bps=COST_BPS)
    mw_summary_rows.append({'strategy': name, **m, 'n_weeks': len(returns_gross)})
    cost_per_wk = (COST_BPS / 10_000.0) * turnover
    for d, rg, tu in zip(r['dates'], r['returns_gross'], r['turnover']):
        mw_weekly_rows.append({'strategy': name, 'date': d,
                               'return_gross': float(rg),
                               'return_net': float(rg - (COST_BPS / 10_000.0) * tu),
                               'turnover': float(tu)})

mw_summary_df = pd.DataFrame(mw_summary_rows).sort_values('sharpe', ascending=False).reset_index(drop=True)
mw_weekly_df = pd.DataFrame(mw_weekly_rows)

display_cols = ['strategy', 'n_weeks', 'total_return_net', 'annualized_return',
                'annualized_vol', 'sharpe', 'sortino', 'max_drawdown', 'calmar',
                'hit_rate', 'avg_turnover']
print('=== full-period (2009-2024) backtest summary, sorted by Sharpe ===')
print(mw_summary_df[display_cols].round(4).to_string(index=False))

# Persist.
MW_OUT_DIR = repo_root() / 'artifacts' / 'backtest' / 'full_period'
MW_OUT_DIR.mkdir(parents=True, exist_ok=True)
mw_summary_df.to_csv(MW_OUT_DIR / 'summary_table.csv', index=False)
mw_weekly_df.to_parquet(MW_OUT_DIR / 'weekly_returns.parquet', compression='zstd', index=False)
print(f'\nwrote -> {MW_OUT_DIR.relative_to(repo_root())}/')

In [ ]:
# Equity curves over full period.
fig, ax = plt.subplots(figsize=(12, 6))
def _color(name):
    if 'NoText' in name: return 'tomato' if 'PPO' in name else 'orange'
    if 'PPO' in name: return 'darkblue'
    if name == 'EW-Top30': return 'gray'
    if name == 'Score-Prop': return 'green'
    if name == 'MinVar': return 'purple'
    if '60/40' in name: return 'brown'
    return 'black'

for name, r in multi_walk_results.items():
    rg = np.array(r['returns_gross'])
    tu = np.array(r['turnover'])
    rn = rg - (COST_BPS / 10_000.0) * tu
    eq = np.cumprod(1.0 + rn)
    ls = '--' if 'NoText' in name else '-'
    ax.plot(r['dates'], eq, label=name, color=_color(name), linestyle=ls, lw=1.6,
            alpha=0.85)
ax.axhline(1.0, color='black', lw=0.5, ls=':', alpha=0.4)
ax.set_yscale('log')
ax.set_xlabel('date')
ax.set_ylabel('cumulative net return (log scale)')
ax.set_title(f'16-year OOS equity curves (2009-2024), {COST_BPS} bps execution cost')
ax.legend(loc='upper left', fontsize=9, ncol=2)
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.savefig(MW_OUT_DIR / 'equity_curves_full_period.png', dpi=120)
plt.show()

# Sharpe bar chart, sorted.
fig, ax = plt.subplots(figsize=(10, 5))
sharpes = mw_summary_df.set_index('strategy')['sharpe']
colors = [_color(s) for s in sharpes.index]
ax.bar(range(len(sharpes)), sharpes.values, color=colors)
ax.set_xticks(range(len(sharpes)))
ax.set_xticklabels(sharpes.index, rotation=30, ha='right')
ax.axhline(0, color='black', lw=0.5)
ax.set_ylabel('annualized Sharpe (rf=0)')
ax.set_title('Sharpe across strategies — full 16-year OOS')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(MW_OUT_DIR / 'sharpe_bars_full_period.png', dpi=120)
plt.show()

# Drawdown chart.
fig, ax = plt.subplots(figsize=(12, 4.5))
for name, r in multi_walk_results.items():
    rg = np.array(r['returns_gross'])
    tu = np.array(r['turnover'])
    rn = rg - (COST_BPS / 10_000.0) * tu
    eq = np.cumprod(1.0 + rn)
    dd = eq / np.maximum.accumulate(eq) - 1.0
    ls = '--' if 'NoText' in name else '-'
    ax.plot(r['dates'], dd, label=name, color=_color(name), linestyle=ls, lw=1.2,
            alpha=0.85)
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('date')
ax.set_ylabel('drawdown')
ax.set_title('Drawdown curves — full 16-year OOS')
ax.legend(loc='lower left', fontsize=8, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(MW_OUT_DIR / 'drawdown_full_period.png', dpi=120)
plt.show()

print(f'\nfigures saved to {MW_OUT_DIR.relative_to(repo_root())}/')